# Tutoriel — Multithreading en Python (pas à pas, FR)
**Date :** 2025-09-27  
**Cible :** Google Colab ou local (PyCharm/VS Code Jupyter)

Ce tutoriel est **guidé** et **exécutable**. À chaque étape :
1. Lis l’énoncé (court).
2. Exécute la cellule **À vous de jouer** (complète les `TODO` si besoin).
3. Vérifie avec la cellule **Solution**.

> Rappel : En Python, les threads n’accélèrent pas les tâches **CPU-bound** à cause du **GIL**. Ils sont utiles pour les **I/O** (attentes réseau, disque, `sleep`) et la **coordination**.

In [ ]:
# Étape 0 — Vérification d'environnement (exécute-moi d'abord)
import sys, platform, threading, time
print("Python :", sys.version.split()[0])
print("Plateforme :", platform.platform())
print("Threads actifs (au départ) :", threading.active_count())

## Étape 1 — Démarrer un thread et attendre la fin
Objectif : créer un thread qui affiche un message, puis attendre sa fin avec `join()`.

In [ ]:
# À vous de jouer
import threading

def dire_bonjour(nom):
    # TODO: afficher [<nom>] Bonjour
    pass

t = threading.Thread(target=dire_bonjour, args=("T0",), name="T0")
# TODO: démarrer le thread
# TODO: attendre la fin

In [ ]:
# Solution
import threading

def dire_bonjour(nom):
    print(f"[{nom}] Bonjour")

t = threading.Thread(target=dire_bonjour, args=("T0",), name="T0")
t.start()
t.join()

## Étape 2 — Nommer les threads et lire le thread courant
Objectif : vérifier que le nom donné correspond à `current_thread().name`.

In [ ]:
# À vous de jouer
import threading

def worker(nom):
    # TODO: lire le nom courant et l'afficher avec le nom passé
    pass

t = threading.Thread(target=worker, args=("Alpha",), name="Alpha")
# TODO: start + join

In [ ]:
# Solution
import threading

def worker(nom):
    courant = threading.current_thread().name
    print("nom_passé=", nom, "| courant=", courant)

t = threading.Thread(target=worker, args=("Alpha",), name="Alpha")
t.start(); t.join()

## Étape 3 — Simuler des I/O avec `sleep` et mesurer le temps
Objectif : lancer 5 threads qui dorment un peu puis s’arrêtent, et mesurer le temps total.

In [ ]:
# À vous de jouer
import threading, time, random

def tache(i):
    # TODO: dormir entre 0.1 et 0.3 s, puis afficher "fin i"
    pass

t0 = time.perf_counter()
threads = [threading.Thread(target=tache, args=(i,)) for i in range(5)]
# TODO: démarrer et joindre tous les threads
t1 = time.perf_counter()
print(f"Temps total ~ {t1 - t0:.3f}s")

In [ ]:
# Solution
import threading, time, random

def tache(i):
    time.sleep(random.uniform(0.1, 0.3))
    print(f"fin {i}")

t0 = time.perf_counter()
threads = [threading.Thread(target=tache, args=(i,)) for i in range(5)]
for t in threads: t.start()
for t in threads: t.join()
t1 = time.perf_counter()
print(f"Temps total ~ {t1 - t0:.3f}s")

## Étape 4 — Threads daemon
Objectif : comprendre qu’un thread **daemon** ne bloque pas la fin du programme.

In [ ]:
# À vous de jouer
import threading, time

def spam():
    while True:
        print('.', end='', flush=True)
        time.sleep(0.1)

t = threading.Thread(target=spam)  # TODO: rendre daemon
# TODO: démarrer, dormir 0.3 s dans le main, puis laisser finir le programme

In [ ]:
# Solution
import threading, time

def spam():
    while True:
        print('.', end='', flush=True)
        time.sleep(0.1)

t = threading.Thread(target=spam, daemon=True)
t.start()
time.sleep(0.3)
# Fin du programme => le daemon s'arrête

## Étape 5 — Variable partagée : condition de course (sans Lock)
Objectif : montrer qu’un simple `compteur += 1` n’est pas sûr en concurrence.

In [ ]:
# À vous de jouer
import threading

compteur = 0

def incr():
    global compteur
    for _ in range(50_000):
        # TODO: incrémenter sans lock
        pass

ts = [threading.Thread(target=incr) for _ in range(4)]
# TODO: start + join
print("Sans Lock (probablement faux) :", compteur)

In [ ]:
# Solution
import threading

compteur = 0

def incr():
    global compteur
    for _ in range(50_000):
        compteur += 1

ts = [threading.Thread(target=incr) for _ in range(4)]
[t.start() for t in ts]
[t.join() for t in ts]
print("Sans Lock (probablement faux) :", compteur)

## Étape 6 — Correction avec `Lock`
Objectif : protéger `compteur += 1` avec `threading.Lock()` pour obtenir le bon résultat.

In [ ]:
# À vous de jouer
import threading

compteur = 0
lock = threading.Lock()

def incr():
    global compteur
    for _ in range(50_000):
        # TODO: protéger compteur += 1 avec lock
        pass

ts = [threading.Thread(target=incr) for _ in range(4)]
# TODO: start + join
print("Avec Lock (attendu 200000) :", compteur)

In [ ]:
# Solution
import threading

compteur = 0
lock = threading.Lock()

def incr():
    global compteur
    for _ in range(50_000):
        with lock:
            compteur += 1

ts = [threading.Thread(target=incr) for _ in range(4)]
[t.start() for t in ts]
[t.join() for t in ts]
print("Avec Lock (attendu 200000) :", compteur)

## Étape 7 — `queue.Queue` : producteur/consommateur
Objectif : envoyer 1..10 via un producteur et les lire côté consommateur (sentinelle `None`).

In [ ]:
# À vous de jouer
import threading, queue

q = queue.Queue()

def producteur():
    # TODO: mettre 1..10 puis sentinelle None
    pass

def consommateur():
    while True:
        x = q.get()
        # TODO: if None -> break, sinon afficher
        q.task_done()

tp = threading.Thread(target=producteur)
tc = threading.Thread(target=consommateur)
# TODO: start, join sur queue, join threads

In [ ]:
# Solution
import threading, queue

q = queue.Queue()

def producteur():
    for i in range(1, 11):
        q.put(i)
    q.put(None)

def consommateur():
    while True:
        x = q.get()
        if x is None:
            q.task_done()
            break
        print("consomme", x)
        q.task_done()

tp = threading.Thread(target=producteur)
tc = threading.Thread(target=consommateur)
tp.start(); tc.start()
q.join()
tp.join(); tc.join()

## Étape 8 — `ThreadPoolExecutor.map` (simplifier le parallélisme I/O)
Objectif : calculer les carrés de 1..10 avec une petite pause, en parallèle.

In [ ]:
# À vous de jouer
from concurrent.futures import ThreadPoolExecutor
import time

def carre(n):
    # TODO: petite pause puis retourner n*n
    pass

with ThreadPoolExecutor(max_workers=4) as ex:
    # TODO: utiliser ex.map sur range(1,11) et convertir en liste
    pass

In [ ]:
# Solution
from concurrent.futures import ThreadPoolExecutor
import time

def carre(n):
    time.sleep(0.05)
    return n*n

with ThreadPoolExecutor(max_workers=4) as ex:
    res = list(ex.map(carre, range(1, 11)))
print(res)

## Étape 9 — `Event` : arrêt propre d’un thread
Objectif : boucler tant qu’un `Event` n’est pas déclenché, puis s’arrêter proprement.

In [ ]:
# À vous de jouer
import threading, time

stop = threading.Event()
ticks = []

def worker():
    # TODO: tant que stop non set: append "tick", sleep 0.1
    pass

t = threading.Thread(target=worker)
# TODO: start, dormir 0.5s, stop.set(), join, afficher len(ticks)

In [ ]:
# Solution
import threading, time

stop = threading.Event()
ticks = []

def worker():
    while not stop.is_set():
        ticks.append("tick")
        time.sleep(0.1)

t = threading.Thread(target=worker)
t.start()
time.sleep(0.5)
stop.set()
t.join()
print("ticks:", len(ticks))

## Étape 10 — `Semaphore` : limiter l’accès (2 max en section critique)
Objectif : s’assurer qu’au plus 2 threads entrent en même temps dans la section critique.

In [ ]:
# À vous de jouer
import threading, time

sem = threading.Semaphore(2)
en_critique = 0
verrou = threading.Lock()

def tache(i):
    global en_critique
    # TODO: with sem: +1 protégé, assert <=2, sleep 0.2, -1 protégé
    pass

threads = [threading.Thread(target=tache, args=(i,)) for i in range(6)]
# TODO: start + join

In [ ]:
# Solution
import threading, time

sem = threading.Semaphore(2)
en_critique = 0
verrou = threading.Lock()

def tache(i):
    global en_critique
    with sem:
        with verrou:
            en_critique += 1
            assert en_critique <= 2
        time.sleep(0.2)
        with verrou:
            en_critique -= 1
    print(i, "ok")

threads = [threading.Thread(target=tache, args=(i,)) for i in range(6)]
for t in threads: t.start()
for t in threads: t.join()

### Fin — Bravo !
Tu peux revenir sur n’importe quelle étape et expérimenter. Pour aller plus loin : `multiprocessing` (CPU-bound) et `asyncio` (I/O non bloquante).